# Sparsity and Time Symmetry: How They Interact

The `time_symmetry` notebook demonstrates time-reversal involutions, and the `sparsification_demo` 
notebook shows how to prune edges from dense rate matrices. This notebook brings the two together.

Key question: **how does sparsification interact with time-reversal asymmetry?**

`sparsify()` removes edges symmetrically (both $i \to j$ and $j \to i$), but the time-reversal 
involution demands its own symmetry: if $s \to s'$ is forbidden, then $\mathrm{inv}(s') \to \mathrm{inv}(s)$ 
must be too. This means **sparse + time-asymmetric systems may lose more edges than intended**, 
since the involution kills additional transitions that sparsification didn't target.

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))

from ctmc import ContinuousTimeMarkovChain as MC
from ctmc import arrhenius_pump_generator, sparsify, is_irreducible
from ctmc import HAS_JAX

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print(f"JAX available: {HAS_JAX}")

## Section A: Single-System Demo

Start with one 15-state Arrhenius pump system. Generate all four variants:
(dense / sparse) $\times$ (time-even / time-asymmetric), and visualize the connectivity.

In [ ]:
np.random.seed(42)
S_demo = 15

R_dense = arrhenius_pump_generator(S=S_demo, N=1, n_pumps=50, pump_strength=5)[0]
R_sparse = sparsify(R_dense, avg_degree=4, seed=42)

# Build all 4 machines
configs = {
    'Dense, time-even':   (R_dense.copy(),  True),
    'Dense, time-asym':   (R_dense.copy(),  False),
    'Sparse, time-even':  (R_sparse.copy(), True),
    'Sparse, time-asym':  (R_sparse.copy(), False),
}

machines = {}
for label, (R, time_even) in configs.items():
    m = MC(R=R)
    if not time_even:
        m.time_even_states = False
        m.set_rate_matrix(m.R)
    machines[label] = m

In [ ]:
# 2x2 connectivity grid
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, (label, m) in zip(axes.ravel(), machines.items()):
    R_vis = m.R.copy()
    np.fill_diagonal(R_vis, 0)
    connectivity = (np.abs(R_vis) > 0).astype(float)
    n_edges = int(connectivity.sum())
    connectivity[connectivity == 0] = np.nan
    
    ax.imshow(connectivity, cmap='Blues', vmin=0, vmax=1, aspect='equal')
    ax.set_title(f'{label}\n{n_edges} edges')
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f'Transition connectivity ({S_demo} states)', fontsize=14)
plt.tight_layout()

In [ ]:
# EPR comparison table
print(f"{'Config':<25s} {'EPR(NESS)':>12s} {'EPR(MEPS)':>12s} {'Excess %':>10s} {'Edges':>6s}")
print('-' * 70)

for label, m in machines.items():
    ness = m.get_ness(force_analytic=True)
    meps = m.get_meps_jax()
    epr_n = m.get_epr(ness)
    epr_m = m.get_epr(meps)
    
    R_off = m.R.copy()
    np.fill_diagonal(R_off, 0)
    n_edges = int((np.abs(R_off) > 0).sum())
    
    excess = (epr_n / epr_m - 1) * 100 if epr_m > 1e-14 else float('nan')
    print(f'{label:<25s} {epr_n:12.6f} {epr_m:12.6f} {excess:9.2f}% {n_edges:6d}')

## Section B: Batch Sweep — Dense vs Sparse $\times$ Time-Even vs Time-Asymmetric

Sweep across system sizes using `arrhenius_pump_generator` (fully connected) as the base,
with `sparsify(avg_degree=4)` for the sparse variant. All four combinations are tested.

In [ ]:
def data_from_machine(p):
    """Collect EPR and DKL statistics from a (possibly batched) machine."""
    unif = p.get_uniform()
    ness = p.get_ness(force_analytic=True)
    meps = p.get_meps_jax()
    return {
        'ness': p.get_epr(ness),
        'meps': p.get_epr(meps),
        'unif': p.get_epr(unif),
        'rand': p.get_epr(p.get_random_state()),
        'nm_dkl': p.dkl(ness, meps),
        'nu_dkl': p.dkl(ness, unif),
        'mu_dkl': p.dkl(meps, unif),
    }

In [ ]:
sizes = [3, 9, 27, 81, 243]
trials = [200, 200, 200, 100, 50]
avg_degree_target = 4

config_labels = ['Dense+Even', 'Dense+Asym', 'Sparse+Even', 'Sparse+Asym']
all_outputs = {label: {} for label in config_labels}

for s, trial in zip(sizes, trials):
    print(f'S={s}')
    n_pumps = max(1, int(0.8 * (s**2 - s) / 2))
    R_dense = arrhenius_pump_generator(S=s, N=trial, n_pumps=n_pumps, pump_strength=4)
    
    # Clamp avg_degree to valid range for small S
    k = min(avg_degree_target, s - 1)
    R_sparse = sparsify(R_dense, avg_degree=k, ensure_connected=True, seed=123)
    
    for R_batch, sparsity_label in [(R_dense, 'Dense'), (R_sparse, 'Sparse')]:
        # Time-even
        p = MC(R=R_batch.copy())
        all_outputs[f'{sparsity_label}+Even'][f'{s}'] = data_from_machine(p)
        
        # Time-asymmetric
        p_asym = MC(R=R_batch.copy())
        p_asym.time_even_states = False
        p_asym.set_rate_matrix(p_asym.R)
        all_outputs[f'{sparsity_label}+Asym'][f'{s}'] = data_from_machine(p_asym)

In [ ]:
# 4-row scatter plot: one row per config, columns = system sizes
nstates = [f'{s}' for s in sizes]

fig, axs = plt.subplots(4, len(sizes), figsize=(3*len(sizes), 12), sharex=True, sharey='row')

for i, label in enumerate(config_labels):
    for j, n in enumerate(nstates):
        ax = axs[i, j]
        data = all_outputs[label][n]
        meps_epr = data['meps']
        
        for state_type in ['ness', 'rand', 'unif']:
            excess = data[state_type] / meps_epr - 1
            valid = np.isfinite(excess) & (excess > 0)
            ax.scatter(meps_epr[valid], excess[valid], label=state_type, alpha=0.3, s=10)
        
        ax.set_yscale('log')
        if i == 0:
            ax.set_title(f'S={n}')
    
    axs[i, 0].set_ylabel(f'{label}\n$\\sigma/\\sigma_{{MEPS}}-1$')

axs[-1, 0].set_xlabel('$\\sigma_{MEPS}$')
axs[-1, -1].legend(markerscale=2)
fig.suptitle('2x2 Comparison: Sparsity x Time Symmetry', fontsize=14, y=1.01)
plt.tight_layout()

In [ ]:
# Summary: median excess EPR vs system size for all 4 configs
fig, ax = plt.subplots(figsize=(9, 6))

style_map = {
    'Dense+Even':  ('tab:blue',   'o', '-'),
    'Dense+Asym':  ('tab:blue',   's', '--'),
    'Sparse+Even': ('tab:orange', 'o', '-'),
    'Sparse+Asym': ('tab:orange', 's', '--'),
}

for label in config_labels:
    color, marker, ls = style_map[label]
    medians = []
    for s in sizes:
        data = all_outputs[label][f'{s}']
        excess = data['ness'] / data['meps'] - 1
        excess = excess[np.isfinite(excess) & (excess > 0)]
        medians.append(np.median(excess) if len(excess) > 0 else np.nan)
    ax.plot(sizes, medians, marker=marker, ls=ls, c=color, label=label, markersize=6)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of states $S$')
ax.set_ylabel('Median $(\\sigma_{NESS}/\\sigma_{MEPS} - 1)$')
ax.set_title('Scaling of Excess EPR: Sparsity x Time Symmetry')
ax.legend()
plt.tight_layout()

## Section C: Sweep Over Sparsity Level

At a fixed system size (S=27), vary the average degree from near-complete down to barely connected.
Compare time-even vs time-asymmetric at each sparsity level to see whether sparsity amplifies,
suppresses, or is orthogonal to the time-asymmetry effect.

In [ ]:
S_fixed = 27
N_fixed = 200
degrees_sweep = [S_fixed - 1, 16, 8, 4, 2]

n_pumps = max(1, int(0.8 * (S_fixed**2 - S_fixed) / 2))
R_base = arrhenius_pump_generator(S=S_fixed, N=N_fixed, n_pumps=n_pumps, pump_strength=4)

sparsity_results = {'even': {}, 'asym': {}}

for k in degrees_sweep:
    print(f'avg_degree={k}')
    if k >= S_fixed - 1:
        R_batch = R_base.copy()
    else:
        R_batch = sparsify(R_base, avg_degree=k, ensure_connected=True, seed=42)
    
    # Time-even
    p = MC(R=R_batch.copy())
    sparsity_results['even'][k] = data_from_machine(p)
    
    # Time-asymmetric
    p2 = MC(R=R_batch.copy())
    p2.time_even_states = False
    p2.set_rate_matrix(p2.R)
    sparsity_results['asym'][k] = data_from_machine(p2)

In [ ]:
# Side-by-side box plots: excess EPR at each sparsity level
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

titles = {'even': 'Time-even', 'asym': 'Time-asymmetric'}

for ax, (sym_label, sym_data) in zip(axes, sparsity_results.items()):
    box_data = []
    box_labels = []
    for k in degrees_sweep:
        excess = sym_data[k]['ness'] / sym_data[k]['meps'] - 1
        excess = excess[np.isfinite(excess) & (excess > 0)]
        box_data.append(excess)
        box_labels.append(f'k={k}' if k < S_fixed - 1 else 'dense')
    
    bp = ax.boxplot(box_data, patch_artist=True, showfliers=True,
                    flierprops=dict(marker='o', markersize=3, alpha=0.4),
                    labels=box_labels)
    
    cmap = plt.cm.viridis_r
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(cmap(i / len(bp['boxes'])))
        patch.set_alpha(0.7)
    
    ax.set_yscale('log')
    ax.set_xlabel('Average degree')
    ax.set_title(titles[sym_label])

axes[0].set_ylabel('$\\sigma_{NESS}/\\sigma_{MEPS} - 1$')
fig.suptitle(f'Excess EPR vs. Sparsity Level (S={S_fixed}, N={N_fixed})', fontsize=14, y=1.02)
plt.tight_layout()

In [ ]:
# Overlay: median excess vs avg_degree for both symmetry types
fig, ax = plt.subplots(figsize=(8, 5))

for sym_label, color, marker in [('even', 'tab:blue', 'o'), ('asym', 'tab:orange', 's')]:
    medians = []
    for k in degrees_sweep:
        excess = sparsity_results[sym_label][k]['ness'] / sparsity_results[sym_label][k]['meps'] - 1
        excess = excess[np.isfinite(excess) & (excess > 0)]
        medians.append(np.median(excess) if len(excess) > 0 else np.nan)
    ax.plot(degrees_sweep, medians, f'{marker}-', c=color, label=f'time-{sym_label}', markersize=8)

ax.set_xlabel('Average degree $k$')
ax.set_ylabel('Median $\\sigma_{NESS}/\\sigma_{MEPS} - 1$')
ax.set_yscale('log')
ax.invert_xaxis()
ax.set_title(f'How Sparsity Modulates the Time Symmetry Effect (S={S_fixed})')
ax.legend()
plt.tight_layout()

## Takeaways

- **Edge loss compounds**: sparse + time-asymmetric systems lose more edges than either effect alone,
  because the involution enforces additional forbidden-transition symmetry on top of the sparsification mask.
- The MEPS principle ($\sigma_{NESS} \geq \sigma_{MEPS}$) holds across all four regimes.
- The excess EPR scaling with system size provides insight into whether sparsity and time asymmetry
  interact synergistically or independently.